In [1]:
print("ahmed")

ahmed


In [2]:
from git import Repo

from langchain_classic.text_splitter import Language
from langchain_classic.document_loaders.generic import GenericLoader
from langchain_classic.document_loaders.parsers import LanguageParser
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import ConversationalRetrievalChain

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from langchain_openai import OpenAI



/home/ahmed/miniconda3/envs/ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
%pwd

'/home/ahmed/CV_ADD_PROJECT/AIEnginear/-Source-Code-Analysis-RAG/research'

## Repo loader

In [4]:
! mkdir input_repo

In [5]:
repo_path = "input_repo/"

Repo.clone_from(url='https://github.com/Ahmed2797/Network-Security.git',to_path=repo_path)

<git.repo.base.Repo '/home/ahmed/CV_ADD_PROJECT/AIEnginear/-Source-Code-Analysis-RAG/research/input_repo/.git'>

## Document loader

In [6]:
path = 'input_repo/Network_Security'

loader = GenericLoader.from_filesystem(path=path,
                                       glob="**/*",
                                       suffixes=['.py'],
                                       parser=LanguageParser(language=Language.PYTHON,
                                                             parser_threshold=100))

In [7]:
documnets = loader.load()

In [12]:
documnets[5].page_content[:200]

"import os\nimport numpy as np \nfrom datetime import date\n\n# MongoDB\nDATA_BASE_NAME = 'NETWORK_SECURITY'\nCOLLECTION_NAME = 'NETWORK_DATA'\nMONGOBD_URL = 'MONGODB_URL'\n\n# Artifacts\nARTIFACTS = 'artifacts'"

In [13]:
document_spliter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,chunk_size = 50,chunk_overlap = 20
)

In [14]:
document_spliter

In [15]:
text_chunk = document_spliter.split_documents(documnets)

In [16]:
print("text_chunk:",len(text_chunk))

text_chunk: 1311


## Embedding Model

In [17]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings
load_dotenv()

# Setup API Keys
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')


In [18]:
embeddings=OpenAIEmbeddings(disallowed_special=(),api_key=OPENAI_API_KEY)


In [19]:
# vector_store = FAISS.from_documents(text_chunk,embeddings)
# ## vector_store.persist()
# vector_store.save_local("faiss_index")

# Initialize Pinecone Client
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "source-code-analysis-rag"

# 1. Check and Create Index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

In [21]:
from langchain_pinecone import PineconeVectorStore

vectorstore = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embeddings,
    index_name=index_name
)

## LLM

In [22]:
from langchain_openai import OpenAI

llm = OpenAI(model="gpt-4.1-nano", temperature=0.2, api_key=OPENAI_API_KEY)

In [23]:
memory = ConversationSummaryMemory(llm=llm,memory_key='chat_history',return_messages=True)

/tmp/ipykernel_61998/4101383306.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(llm=llm,memory_key='chat_history',return_messages=True)


In [24]:
retriver = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 50})
search = ConversationalRetrievalChain.from_llm(llm=llm,retriever=retriver,memory=memory)

In [25]:
retriver

VectorStoreRetriever(tags=['PineconeVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x75e02edafdc0>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 50})

In [26]:
search

ConversationalRetrievalChain(memory=ConversationSummaryMemory(llm=OpenAI(client=<openai.resources.completions.Completions object at 0x75e02f262860>, async_client=<openai.resources.completions.AsyncCompletions object at 0x75def88e3d30>, model_name='gpt-4.1-nano', temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********')), chat_memory=InMemoryChatMessageHistory(messages=[]), return_messages=True, memory_key='chat_history'), verbose=False, combine_docs_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"), llm=OpenAI(client=<openai.resources.completions.Completions object at 0x75e02f262860>, async_client=<openai.resources.complet

In [27]:
question = 'whats get_feature_extract_data'
result = search(question)
result

/tmp/ipykernel_61998/329157500.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = search(question)


{'question': 'whats get_feature_extract_data',
 'chat_history': [SystemMessage(content='', additional_kwargs={}, response_metadata={})],
 'answer': ' The function get_feature_extract_data appears to be a method that returns an object named data_transformation_artifact, which likely contains data related to feature extraction. It is part of a class, as indicated by the use of self. The method also logs an informational message about extracting a DataFrame, suggesting it is involved in data processing or transformation tasks. However, the exact details of what data it returns or how it functions are not provided in the snippet.'}

In [28]:
result['answer']

' The function get_feature_extract_data appears to be a method that returns an object named data_transformation_artifact, which likely contains data related to feature extraction. It is part of a class, as indicated by the use of self. The method also logs an informational message about extracting a DataFrame, suggesting it is involved in data processing or transformation tasks. However, the exact details of what data it returns or how it functions are not provided in the snippet.'